# Toy Store E-Commerce: A/B Test Analysis
## Billing Page & Landing Page Conversion Rate Comparisons

**Tools:** Python, scipy, numpy  
**Dataset:** [Maven Analytics — Toy Store E-Commerce Database](https://mavenanalytics.io/data-playground)  
**Project:** Toy Store E-Commerce Funnel Analysis

---

This notebook contains two chi-square tests of independence examining whether observed differences in conversion rates across landing page and billing page variants are statistically significant, or attributable to random variation.

All session counts and conversion counts were derived from BigQuery SQL queries on the `toystore` dataset. See the project SQL files for full query details.


## Setup

In [ ]:
from scipy import stats
from itertools import combinations
import numpy as np

---
## Test 1: Billing Page Redesign (`/billing` vs `/billing-2`)

### Background
The toy store ran two versions of its billing/checkout page over the course of the dataset:
- `/billing` — the original billing page, used primarily in the earlier period
- `/billing-2` — a redesigned billing page, introduced later

A session was counted as a conversion if it reached the `/thank-you-for-your-order` confirmation page.

### Hypothesis
- **H₀ (Null):** There is no difference in conversion rate between `/billing` and `/billing-2`
- **H₁ (Alternative):** `/billing-2` has a different conversion rate than `/billing`
- **Significance level:** α = 0.05

### Limitation
The two variants were not run concurrently — `/billing` was active primarily in the earlier period, `/billing-2` later. Sample sizes are also substantially unbalanced (3,617 vs 48,441 sessions). This is an observational comparison rather than a randomized controlled experiment, so the result reflects strong association, not proven causation.


In [ ]:
billing_sessions = 3617
billing_two_sessions = 48441
billing_conversions = 1620
billing_two_conversions = 30693

contingency_table = np.array([
    [billing_conversions, billing_sessions - billing_conversions],
    [billing_two_conversions, billing_two_sessions - billing_two_conversions]
])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

billing_cvr = billing_conversions / billing_sessions * 100
billing_two_cvr = billing_two_conversions / billing_two_sessions * 100
lift = billing_two_cvr - billing_cvr

print("=" * 45)
print("  Billing Page A/B Test Results")
print("=" * 45)
print(f"  /billing CVR:    {billing_cvr:.2f}%  (n={billing_sessions:,})")
print(f"  /billing-2 CVR:  {billing_two_cvr:.2f}%  (n={billing_two_sessions:,})")
print(f"  Lift:            +{lift:.2f} percentage points")
print("-" * 45)
print(f"  Chi-square:      {chi2:.4f}")
print(f"  P-value:         {p_value:.2e}")
print(f"  Degrees of freedom: {dof}")
print("=" * 45)


### Interpretation

The difference between `/billing` and `/billing-2` is **statistically significant** (p < 0.001). We reject the null hypothesis.

`/billing-2` is associated with an 18.57 percentage point improvement in checkout completion rate (44.79% → 63.36%), suggesting the redesigned billing page meaningfully improved conversion.

**Recommendation:** Treat `/billing-2` as the preferred billing page design. A properly randomized follow-up test would be needed to confirm causation before attributing the improvement solely to the page redesign.


---
## Test 2: Landing Page Iterations (`/home` through `/lander-5`)

### Background
The site launched with a single `/home` entry page in March 2012. Five additional landing page variants were introduced sequentially through August 2014:

| Page | First Seen |
|------|------------|
| `/home` | March 2012 |
| `/lander-1` | June 2012 |
| `/lander-2` | January 2013 |
| `/lander-3` | July 2013 |
| `/lander-4` | February 2014 |
| `/lander-5` | August 2014 |

A session was counted as a conversion if it reached the `/thank-you-for-your-order` confirmation page.

### Hypothesis
- **H₀ (Null):** Conversion rates are equal across all six entry page variants
- **H₁ (Alternative):** At least one variant has a meaningfully different conversion rate
- **Significance level:** α = 0.05


In [ ]:
# [conversions, non-conversions] per variant
data = {
    'home':     [9711,  127865],
    'lander_1': [2157,  45417],
    'lander_2': [10128, 121042],
    'lander_3': [2679,  76321],
    'lander_4': [708,   8677],
    'lander_5': [6930,  61236]
}

contingency_table = np.array(list(data.values()))
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print("=" * 50)
print("  Lander Comparison — Overall Chi-Square Test")
print("=" * 50)
print(f"  Chi-square:         {chi2:.4f}")
print(f"  P-value:            {p_value:.2e}")
print(f"  Degrees of freedom: {dof}")
print("=" * 50)
print()

# Print CVR per variant
print(f"  {'Variant':<12} {'Sessions':>10} {'Conversions':>13} {'CVR':>8}")
print("  " + "-" * 46)
for variant, (conv, non_conv) in data.items():
    sessions = conv + non_conv
    cvr = conv / sessions * 100
    print(f"  {variant:<12} {sessions:>10,} {conv:>13,} {cvr:>7.2f}%")


### Interpretation

The overall chi-square test is **statistically significant** (p < 0.001). We reject the null hypothesis — conversion rates are not equal across all six variants.

However, a significant omnibus result only tells us that *something* differs across the group. It does not tell us *which specific pairs* of variants are meaningfully different from each other. Post-hoc pairwise testing is required.


---
## Test 3: Post-Hoc Pairwise Comparisons (Bonferroni Correction)

### Why Bonferroni Correction?
With 6 variants there are 15 possible pairwise comparisons. Running 15 separate chi-square tests at α = 0.05 inflates the family-wise Type I error rate — the probability of finding at least one false positive by chance alone rises to ~54%. 

The **Bonferroni correction** adjusts the significance threshold by dividing α by the number of comparisons:

> Corrected α = 0.05 / 15 = **0.0033**

Any pairwise p-value above 0.0033 is treated as not statistically significant.


In [ ]:
variants = list(data.keys())
results = []

for v1, v2 in combinations(variants, 2):
    table = np.array([data[v1], data[v2]])
    chi2_pair, p_pair, _, _ = stats.chi2_contingency(table)
    
    cvr1 = data[v1][0] / sum(data[v1]) * 100
    cvr2 = data[v2][0] / sum(data[v2]) * 100
    
    results.append((v1, v2, p_pair, cvr1, cvr2))

alpha_corrected = 0.05 / 15

print(f"Bonferroni-corrected significance threshold: {alpha_corrected:.4f}")
print()
print(f"  {'Comparison':<30} {'CVR 1':>8} {'CVR 2':>8} {'P-value':>12}  {'Result'}")
print("  " + "-" * 75)

for v1, v2, p, cvr1, cvr2 in sorted(results, key=lambda x: x[2]):
    sig = "✓ significant" if p < alpha_corrected else "✗ not significant"
    print(f"  {v1+' vs '+v2:<30} {cvr1:>7.2f}% {cvr2:>7.2f}% {p:>12.4e}  {sig}")


### Interpretation

**13 of 15 pairwise comparisons are statistically significant** after Bonferroni correction.

**The two non-significant pairs are the most actionable findings:**

- **`/home` vs `/lander-4`** (p = 0.0798): `/lander-4` failed to improve on the original homepage. This iteration was essentially a failed experiment — no measurable conversion lift was achieved despite the design change.
- **`/lander-2` vs `/lander-4`** (p = 0.5471): `/lander-4` also could not beat `/lander-2`, a version introduced over a year earlier. This represents a step backward in the optimization process.

**The broader pattern:**

Lander optimization was **non-linear**. Conversion rates did not simply improve with each new version:

| Variant | CVR | vs /home |
|---------|-----|----------|
| /home | 7.06% | baseline |
| /lander-1 | 4.53% | ↓ worse |
| /lander-2 | 7.72% | ↑ better |
| /lander-3 | 3.39% | ↓ worse |
| /lander-4 | 7.54% | ↔ no difference |
| /lander-5 | 10.17% | ↑ significantly better |

`/lander-5` is the clear winner — achieving a **44% relative lift** over the original homepage (7.06% → 10.17%), the only version to deliver a statistically significant and meaningfully sized improvement.

**Recommendation:** Use `/lander-5` as the reference design for future landing page work. The non-linear results across iterations suggest that A/B testing each change incrementally (rather than bundling multiple changes per iteration) would make it easier to identify which specific elements drive conversion improvement.
